# 04 — The Marathi Severity Inversion

A metric that is merely noisy under romanisation would degrade uniformly. Marathi
degrades *systematically in the wrong direction*: the worse a segment's annotated
error, the more romanisation **raises** its COMET score.

Δ is defined throughout as `COMET_romanised − COMET_native`, so a positive Δ means
romanisation rewarded the segment.

**Base:** 1,258 Marathi segments carrying a non-`Default` severity label
(the Wilcoxon test is reported over all 1,400).

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/severity_inversion.csv`

## Step 0 — Configuration

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Step 1 — Load the Workbook

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

SEVERITY_ORDER = ["Very Low", "Low", "Medium", "High", "Very High"]

## Step 2 — Mean ΔCOMET by Severity Bucket

If COMET were behaving, Δ would be flat or would fall as severity rises. Instead
it climbs monotonically from negative to strongly positive.

In [ ]:
d = full["MAR"].copy()
d["delta"] = d[COL_COMET_ROM] - d[COL_COMET_NAT]

rows = []
print("Marathi — mean \u0394COMET (romanised \u2212 native) by annotated severity")
print(f"{'Severity':>10}  {'N':>5}  {'mean \u0394COMET':>13}")
print("-" * 32)
for s in SEVERITY_ORDER:
    sub = d[d[COL_SEVERITY] == s]
    rows.append(dict(severity=s, N=len(sub), mean_dcomet=sub["delta"].mean()))
    print(f"{s:>10}  {len(sub):>5}  {sub['delta'].mean():>+13.2f}")

buckets = pd.DataFrame(rows).set_index("severity")

assert buckets["N"].sum() == 1258, f"severity base = {buckets['N'].sum()}, expected 1258"
assert buckets["mean_dcomet"].is_monotonic_increasing, "\u0394COMET should rise with severity"
assert buckets.loc["Very Low", "mean_dcomet"] < 0 < buckets.loc["Very High", "mean_dcomet"]
print(f"\n\u2713 Severity base = {buckets['N'].sum():,} (paper: 1,258)")
print("\u2713 \u0394COMET rises monotonically with severity "
      f"({buckets.loc['Very Low', 'mean_dcomet']:+.2f} \u2192 "
      f"{buckets.loc['Very High', 'mean_dcomet']:+.2f})")

## Step 3 — Sentence-Level Rank Correlation

The bucket means could in principle be driven by a handful of segments. A
sentence-level Spearman ρ over the ordinal severity scale confirms the trend
holds across all 1,258 annotated segments.

In [ ]:
nd = d[d[COL_SEVERITY].isin(SEVERITY_ORDER)]
smap = {s: i for i, s in enumerate(SEVERITY_ORDER)}
rho, p = stats.spearmanr(nd[COL_SEVERITY].map(smap), nd["delta"])
print(f"  severity ~ \u0394COMET:  \u03c1 = {rho:+.3f}   p = {p:.2e}   N = {len(nd):,}")

assert rho > 0 and p < 1e-9
print(f"\n\u2713 Positive and highly significant: worse errors \u2192 larger COMET gain")

## Step 4 — Is the Overall Shift Even Significant?

Over all 1,400 Marathi segments the *median* Δ is slightly negative while the
*mean* is positive — the shift is driven by a heavy right tail, not by a uniform
lift. The Wilcoxon signed-rank test is reported for completeness; the inversion
finding rests on Steps 2 and 3, not on this test.

In [ ]:
w = stats.wilcoxon(d["delta"])
print(f"  Wilcoxon signed-rank over all {len(d):,} segments")
print(f"    W       = {w.statistic:,.0f}")
print(f"    p       = {w.pvalue:.3f}")
print(f"    median  = {d['delta'].median():+.2f}")
print(f"    mean    = {d['delta'].mean():+.2f}")
print("\n  The median is negative and the mean positive: a right-tailed shift, "
      "not a uniform one.")

## Step 5 — The Worked Example

The single most legible case: a segment a human scored 0/25 with a `Very High`
severity omission, which romanisation lifts by 26 COMET points.

In [ ]:
zero = d[d["H"] == 0]
vh = zero[zero[COL_SEVERITY] == "Very High"].sort_values("delta", ascending=False)
r = vh.iloc[0]
print(f"  MQM = 0/25, severity = Very High, error = {r['Error1_Type']}")
print(f"    COMET native    = {r[COL_COMET_NAT]:.1f}")
print(f"    COMET romanised = {r[COL_COMET_ROM]:.1f}")
print(f"    \u0394               = {r['delta']:+.1f}")

assert abs(r[COL_COMET_NAT] - 36.6) < 0.05
assert abs(r[COL_COMET_ROM] - 62.7) < 0.05
assert abs(r["delta"] - 26.1) < 0.05
print(f"\n\u2713 {r[COL_COMET_NAT]:.1f} \u2192 {r[COL_COMET_ROM]:.1f} "
      f"({r['delta']:+.1f}) (paper: 36.6 \u2192 62.7, +26.1)")

## Step 6 — Save

In [ ]:
out = buckets.copy()
out.loc["_spearman_rho"] = [len(nd), rho]
out.loc["_spearman_p"] = [len(nd), p]
out.loc["_wilcoxon_W"] = [len(d), w.statistic]
out.loc["_wilcoxon_p"] = [len(d), w.pvalue]
out.loc["_delta_median"] = [len(d), d["delta"].median()]
out.loc["_delta_mean"] = [len(d), d["delta"].mean()]

path = TABLES_DIR / "severity_inversion.csv"
out.to_csv(path)
print(out.to_string())
print(f"\nSaved \u2192 {path}")

## Step 7 — Output Manifest

In [ ]:
print("=== Notebook 04 — output manifest ===")
print("  severity_inversion.csv")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1